# Figure 6 — Distance vs Similarity Sub-Sampling (SNJ α sweep)

Sub-sample the similarity matrix `M = JC_similarity(obs)` vs the distance matrix `D = paralinear_distance(obs)`, then post-transform `S = exp(-α · D̂)` on the distance branch.

Question: does sampling D then exponentiating recover the same Fiedler partition as sampling M directly?

**Theory:** With IPW debiasing, `E[exp(-α D̂)] ≠ exp(-α D)` because exp is nonlinear and sampling is element-wise. The SNJ paper's α parameter (`spectraltree/snj.py:28`) is the equivalent power on M; see `docs/papers/SNJ_Jaffe_Kluger.pdf`.

**Metric:** σ₂ (second singular value) of the partition cross-block on the canonical similarity-view, per SNJ.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

from src.core.similarity_builder import SimilarityMatrixBuilder
from src.core.utils import compute_fielder_vector, generate_sequences
from src.models.tree_models import get_tree_factory
from src.models.sequence_models import get_sequence_factory
from src.utils.metrics import compute_partition_agreement, compute_reference_partition_and_quality

rng = np.random.default_rng(42)

## Cell 2 — Generate tree + observations once

In [ ]:
N_TAXA = 512        # representative size; the on-disk sweep covers 512..8192
SEQ_LEN = 10000
MUT_RATE = 0.1

tree_factory = get_tree_factory('balanced_binary', {'num_taxa': N_TAXA, 'edge_length': 1.0})
tree = tree_factory()
seq_factory = get_sequence_factory('JC69', {'mutation_rate': MUT_RATE})
seq_model = seq_factory()
observations = generate_sequences(N_TAXA, SEQ_LEN, MUT_RATE, tree_model=tree, seq_model=seq_model)
print('observations:', observations.shape)


## Cell 3 — Sweep matrix_kind × α × p

For each (kind, α), the canonical similarity-view at p=1 is the reference. We compute σ₂ partition agreement vs that reference for several p values, averaged over bootstrap_reps.

In [ ]:
p_values = [0.0005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
alphas = [0.5, 1.0, 2.0]
bootstrap_reps = 5

results = {}  # results[(kind, alpha)] -> list of (p, mean_agreement_pct, mean_sigma2)

for kind, alpha_list in [('similarity', [1.0]), ('distance', alphas)]:
    for alpha in alpha_list:
        builder = SimilarityMatrixBuilder(
            method='uniform', matrix_kind=kind, distance_alpha=alpha
        )
        if kind == 'similarity':
            M_ref = builder.build_full(observations)
        else:
            D_full = builder.build_full(observations)
            M_ref = np.exp(-alpha * D_full); np.fill_diagonal(M_ref, 1.0)

        v_ref = compute_fielder_vector(M_ref)
        part_ref, sigma2_ref, _ = compute_reference_partition_and_quality(v_ref, M_ref, num_gaps=0, min_split=1)

        per_p = []
        for p in p_values:
            agreements, sigma2s = [], []
            for rep in range(bootstrap_reps):
                S_hat = builder.build_subsampled(observations, p, seed=42 + rep)
                v_hat = compute_fielder_vector(S_hat)
                if np.dot(v_hat, v_ref) < 0:
                    v_hat = -v_hat
                # compute_partition_agreement returns agreement already in percent (0-100)
                agreement_M, sigma2_M, _ = compute_partition_agreement(part_ref, v_hat, M_ref)
                agreements.append(agreement_M)
                sigma2s.append(sigma2_M)
            per_p.append((p, float(np.mean(agreements)), float(np.mean(sigma2s))))
            print(f'  {kind:11s} α={alpha:.1f}  p={p:.2f}  agreement={np.mean(agreements):5.1f}%  σ₂={np.mean(sigma2s):.4f}')
        results[(kind, alpha)] = per_p


## Cell 4 — Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for (kind, alpha), rows in results.items():
    ps = [r[0] for r in rows]
    agr = [r[1] for r in rows]  # already in percent
    s2 = [r[2] for r in rows]
    label = f'{kind} α={alpha:.1f}' if kind == 'distance' else 'similarity'
    ls = '--' if kind == 'distance' else '-'
    axes[0].plot(ps, agr, ls, marker='o', label=label)
    axes[1].plot(ps, s2, ls, marker='o', label=label)

axes[0].set_xlabel('sampling rate p'); axes[0].set_ylabel('partition agreement (%)')
axes[0].set_title('Partition agreement vs reference')
axes[0].set_xscale('log'); axes[0].grid(True, alpha=0.3); axes[0].legend()

axes[1].set_xlabel('sampling rate p'); axes[1].set_ylabel(r'$\sigma_2$ (rank-1 deviation)')
axes[1].set_title(r'Cross-block $\sigma_2$ (lower = cleaner clade)')
axes[1].set_xscale('log'); axes[1].grid(True, alpha=0.3); axes[1].legend()

fig.suptitle(f'Figure 6: distance vs similarity sub-sampling (balanced binary, n={N_TAXA}, L={SEQ_LEN}, μ={MUT_RATE})')
fig.tight_layout()
plt.show()
